# Loss Functions for Classification: MSE vs Cross-Entropy vs Label Smoothing

[![Open In Colab](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-loss-functions-classification.ipynb)](https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-loss-functions-classification.ipynb)

## Overview

In this notebook, we'll explore one of the most important choices in training classification models: **which loss function to use**. While cross-entropy is the standard choice for classification, we'll investigate:

1. **Why loss functions matter** - How they shape the learning process
2. **MSE for classification** - Can regression loss work for classification?
3. **Cross-Entropy loss** - The standard choice and why it works
4. **Label Smoothing** - A regularization technique to improve generalization

We'll compare these approaches empirically with visualizations and metrics.

## Setup and Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('mps' if torch.backends.mps.is_available() else 
                      'cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'seaborn'

## Part 1: Mathematical Foundations

### Cross-Entropy Loss

Cross-entropy loss measures the difference between two probability distributions. For classification, it compares the predicted probability distribution with the true distribution (one-hot encoded labels).

**Formula:**
$$L_{CE} = -\sum_{i=1}^{C} y_i \log(\hat{y}_i)$$

Where:
- $C$ = number of classes
- $y_i$ = true label (1 for correct class, 0 otherwise)
- $\hat{y}_i$ = predicted probability for class $i$

**Key properties:**
- Penalizes confident wrong predictions heavily
- Gradient is proportional to prediction error: $\nabla L = \hat{y} - y$
- Well-suited for probabilistic interpretation

### Mean Squared Error (MSE) Loss

MSE measures the average squared difference between predictions and targets.

**Formula:**
$$L_{MSE} = \frac{1}{C}\sum_{i=1}^{C} (y_i - \hat{y}_i)^2$$

**Problems for classification:**
- Gradient vanishes when prediction is confidently wrong: $\nabla L = 2(\hat{y} - y) \cdot \hat{y}(1-\hat{y})$
- Not designed for probability distributions
- Can lead to slower convergence

### Visualizing Loss Functions

Let's plot how these loss functions behave for a binary classification problem.

In [ ]:
# Predicted probabilities for the correct class
probs = np.linspace(0.01, 0.99, 100)

# Cross-entropy loss: -log(p) for correct class
ce_loss = -np.log(probs)

# MSE loss: (1 - p)^2 for correct class
mse_loss = (1 - probs) ** 2

plt.figure(figsize=(14, 5))

# Plot loss values
plt.subplot(1, 2, 1)
plt.plot(probs, ce_loss, label='Cross-Entropy', linewidth=2)
plt.plot(probs, mse_loss, label='MSE', linewidth=2)
plt.xlabel('Predicted Probability (for correct class)', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Loss Function Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

# Plot gradients
plt.subplot(1, 2, 2)
ce_grad = -1 / probs  # Gradient of -log(p)
mse_grad = -2 * (1 - probs)  # Gradient of (1-p)^2

plt.plot(probs, ce_grad, label='Cross-Entropy', linewidth=2)
plt.plot(probs, mse_grad, label='MSE', linewidth=2)
plt.xlabel('Predicted Probability (for correct class)', fontsize=12)
plt.ylabel('Gradient Magnitude', fontsize=12)
plt.title('Gradient Comparison', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(-10, 0)

plt.tight_layout()
plt.show()

print("Key Observations:")
print("1. Cross-entropy grows much larger for confident wrong predictions")
print("2. Cross-entropy gradient is larger when predictions are wrong (left side)")
print("3. MSE gradient is smaller when predictions are confidently wrong")
print("4. This explains why cross-entropy converges faster!")

## Part 2: Binary Classification Experiment

Let's create a simple 2D binary classification problem and train models with both loss functions.

### Generate Synthetic Data

In [ ]:
# Generate moons dataset
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.LongTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.LongTensor(y_test)

# Visualize the data
plt.figure(figsize=(8, 6))
plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', label='Class 0', alpha=0.6, s=30)
plt.scatter(X[y==1, 0], X[y==1, 1], c='red', label='Class 1', alpha=0.6, s=30)
plt.xlabel('Feature 1', fontsize=12)
plt.ylabel('Feature 2', fontsize=12)
plt.title('Binary Classification Dataset (Moons)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

### Define a Simple Neural Network

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, output_dim=2):
        super(SimpleClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)  # No activation, logits
        return x

# Test the model
model = SimpleClassifier()
print(f"Model architecture:\n{model}")
print(f"\nNumber of parameters: {sum(p.numel() for p in model.parameters()):,}")

### Training Function

In [ ]:
def train_model(model, X_train, y_train, X_test, y_test, 
                loss_fn, optimizer, epochs=100, use_softmax=False):
    """
    Train a model and track metrics.
    
    Args:
        use_softmax: If True, apply softmax before computing MSE loss
    """
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        optimizer.zero_grad()
        
        outputs = model(X_train)
        
        # Compute loss based on type
        if use_softmax:
            # For MSE, we need probabilities
            probs = F.softmax(outputs, dim=1)
            y_one_hot = F.one_hot(y_train, num_classes=2).float()
            loss = loss_fn(probs, y_one_hot)
        else:
            # Cross-entropy expects logits
            loss = loss_fn(outputs, y_train)
        
        loss.backward()
        optimizer.step()
        
        # Compute accuracy
        with torch.no_grad():
            train_pred = outputs.argmax(dim=1)
            train_acc = (train_pred == y_train).float().mean().item()
        
        # Evaluation
        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test)
            
            if use_softmax:
                test_probs = F.softmax(test_outputs, dim=1)
                y_test_one_hot = F.one_hot(y_test, num_classes=2).float()
                test_loss = loss_fn(test_probs, y_test_one_hot)
            else:
                test_loss = loss_fn(test_outputs, y_test)
            
            test_pred = test_outputs.argmax(dim=1)
            test_acc = (test_pred == y_test).float().mean().item()
        
        train_losses.append(loss.item())
        test_losses.append(test_loss.item())
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs} - Train Loss: {loss.item():.4f}, "
                  f"Test Loss: {test_loss.item():.4f}, Train Acc: {train_acc:.4f}, "
                  f"Test Acc: {test_acc:.4f}")
    
    return train_losses, test_losses, train_accs, test_accs

### Train with Cross-Entropy Loss

In [ ]:
# Initialize model
model_ce = SimpleClassifier()
optimizer_ce = optim.Adam(model_ce.parameters(), lr=0.01)
loss_fn_ce = nn.CrossEntropyLoss()

print("Training with Cross-Entropy Loss...\n")
train_losses_ce, test_losses_ce, train_accs_ce, test_accs_ce = train_model(
    model_ce, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor,
    loss_fn_ce, optimizer_ce, epochs=100, use_softmax=False
)

### Train with MSE Loss

In [ ]:
# Initialize model
model_mse = SimpleClassifier()
optimizer_mse = optim.Adam(model_mse.parameters(), lr=0.01)
loss_fn_mse = nn.MSELoss()

print("Training with MSE Loss...\n")
train_losses_mse, test_losses_mse, train_accs_mse, test_accs_mse = train_model(
    model_mse, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor,
    loss_fn_mse, optimizer_mse, epochs=100, use_softmax=True
)

### Compare Training Dynamics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Training loss
axes[0, 0].plot(train_losses_ce, label='Cross-Entropy', linewidth=2)
axes[0, 0].plot(train_losses_mse, label='MSE', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel('Loss', fontsize=11)
axes[0, 0].set_title('Training Loss', fontsize=13, fontweight='bold')
axes[0, 0].legend(fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Test loss
axes[0, 1].plot(test_losses_ce, label='Cross-Entropy', linewidth=2)
axes[0, 1].plot(test_losses_mse, label='MSE', linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel('Loss', fontsize=11)
axes[0, 1].set_title('Test Loss', fontsize=13, fontweight='bold')
axes[0, 1].legend(fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Training accuracy
axes[1, 0].plot(train_accs_ce, label='Cross-Entropy', linewidth=2)
axes[1, 0].plot(train_accs_mse, label='MSE', linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('Accuracy', fontsize=11)
axes[1, 0].set_title('Training Accuracy', fontsize=13, fontweight='bold')
axes[1, 0].legend(fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Test accuracy
axes[1, 1].plot(test_accs_ce, label='Cross-Entropy', linewidth=2)
axes[1, 1].plot(test_accs_mse, label='MSE', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=11)
axes[1, 1].set_ylabel('Accuracy', fontsize=11)
axes[1, 1].set_title('Test Accuracy', fontsize=13, fontweight='bold')
axes[1, 1].legend(fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final Results:")
print(f"Cross-Entropy - Train Acc: {train_accs_ce[-1]:.4f}, Test Acc: {test_accs_ce[-1]:.4f}")
print(f"MSE           - Train Acc: {train_accs_mse[-1]:.4f}, Test Acc: {test_accs_mse[-1]:.4f}")

### Visualize Decision Boundaries

In [ ]:
def plot_decision_boundary(model, X, y, title):
    """Plot decision boundary for a 2D classifier."""
    h = 0.02  # Step size in mesh
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Predict on mesh
    model.eval()
    with torch.no_grad():
        Z = model(torch.FloatTensor(np.c_[xx.ravel(), yy.ravel()]))
        Z = Z.argmax(dim=1).numpy()
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', label='Class 0', 
                edgecolors='k', s=40, alpha=0.7)
    plt.scatter(X[y==1, 0], X[y==1, 1], c='red', label='Class 1', 
                edgecolors='k', s=40, alpha=0.7)
    plt.xlabel('Feature 1', fontsize=11)
    plt.ylabel('Feature 2', fontsize=11)
    plt.title(title, fontsize=13, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)

plt.figure(figsize=(16, 6))

plt.subplot(1, 2, 1)
plot_decision_boundary(model_ce, X_test, y_test, 
                       f'Cross-Entropy (Acc: {test_accs_ce[-1]:.3f})')

plt.subplot(1, 2, 2)
plot_decision_boundary(model_mse, X_test, y_test, 
                       f'MSE (Acc: {test_accs_mse[-1]:.3f})')

plt.tight_layout()
plt.show()

## Part 3: Multi-class Classification (MNIST)

Let's test these loss functions on a more realistic problem: MNIST digit classification.

### Load MNIST Dataset

In [ ]:
# Data transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load datasets
train_dataset = torchvision.datasets.MNIST(root='./data', train=True, 
                                           download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, 
                                          download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

# Visualize some samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Label: {label}', fontsize=11)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Define CNN Classifier

In [ ]:
class CNNClassifier(nn.Module):
    def __init__(self, num_classes=10):
        super(CNNClassifier, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.25)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Test the model
model = CNNClassifier()
print(f"Model architecture:\n{model}")
print(f"\nNumber of parameters: {sum(p.numel() for p in model.parameters()):,}")

### Training Function for MNIST

In [ ]:
def train_mnist_model(model, train_loader, test_loader, loss_fn, 
                      optimizer, epochs=10, use_softmax=False, device='cpu'):
    """Train a model on MNIST."""
    model = model.to(device)
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    
    for epoch in range(epochs):
        # Training
        model.train()
        train_loss = 0
        correct = 0
        total = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            outputs = model(data)
            
            if use_softmax:
                probs = F.softmax(outputs, dim=1)
                target_one_hot = F.one_hot(target, num_classes=10).float()
                loss = loss_fn(probs, target_one_hot)
            else:
                loss = loss_fn(outputs, target)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            pred = outputs.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
        
        train_loss /= len(train_loader)
        train_acc = correct / total
        
        # Evaluation
        model.eval()
        test_loss = 0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                outputs = model(data)
                
                if use_softmax:
                    probs = F.softmax(outputs, dim=1)
                    target_one_hot = F.one_hot(target, num_classes=10).float()
                    loss = loss_fn(probs, target_one_hot)
                else:
                    loss = loss_fn(outputs, target)
                
                test_loss += loss.item()
                pred = outputs.argmax(dim=1)
                correct += (pred == target).sum().item()
                total += target.size(0)
        
        test_loss /= len(test_loader)
        test_acc = correct / total
        
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, "
              f"Test Loss: {test_loss:.4f}, Train Acc: {train_acc:.4f}, "
              f"Test Acc: {test_acc:.4f}")
    
    return train_losses, test_losses, train_accs, test_accs

### Train with Cross-Entropy

In [ ]:
model_mnist_ce = CNNClassifier()
optimizer_mnist_ce = optim.Adam(model_mnist_ce.parameters(), lr=0.001)
loss_fn_mnist_ce = nn.CrossEntropyLoss()

print("Training MNIST with Cross-Entropy Loss...\n")
train_losses_mnist_ce, test_losses_mnist_ce, train_accs_mnist_ce, test_accs_mnist_ce = train_mnist_model(
    model_mnist_ce, train_loader, test_loader, loss_fn_mnist_ce, 
    optimizer_mnist_ce, epochs=10, use_softmax=False, device=device
)

### Train with MSE

In [ ]:
model_mnist_mse = CNNClassifier()
optimizer_mnist_mse = optim.Adam(model_mnist_mse.parameters(), lr=0.001)
loss_fn_mnist_mse = nn.MSELoss()

print("Training MNIST with MSE Loss...\n")
train_losses_mnist_mse, test_losses_mnist_mse, train_accs_mnist_mse, test_accs_mnist_mse = train_mnist_model(
    model_mnist_mse, train_loader, test_loader, loss_fn_mnist_mse, 
    optimizer_mnist_mse, epochs=10, use_softmax=True, device=device
)

### Compare MNIST Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Test loss
axes[0].plot(test_losses_mnist_ce, label='Cross-Entropy', linewidth=2, marker='o')
axes[0].plot(test_losses_mnist_mse, label='MSE', linewidth=2, marker='s')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('MNIST Test Loss Comparison', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Test accuracy
axes[1].plot(test_accs_mnist_ce, label='Cross-Entropy', linewidth=2, marker='o')
axes[1].plot(test_accs_mnist_mse, label='MSE', linewidth=2, marker='s')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('MNIST Test Accuracy Comparison', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal MNIST Results:")
print(f"Cross-Entropy - Test Acc: {test_accs_mnist_ce[-1]:.4f}")
print(f"MSE           - Test Acc: {test_accs_mnist_mse[-1]:.4f}")

### Analyze Prediction Confidence

In [ ]:
def get_confidence_distribution(model, test_loader, device):
    """Get distribution of prediction confidences."""
    model.eval()
    confidences = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data = data.to(device)
            outputs = model(data)
            probs = F.softmax(outputs, dim=1)
            max_probs, _ = probs.max(dim=1)
            confidences.extend(max_probs.cpu().numpy())
    
    return np.array(confidences)

conf_ce = get_confidence_distribution(model_mnist_ce, test_loader, device)
conf_mse = get_confidence_distribution(model_mnist_mse, test_loader, device)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.hist(conf_ce, bins=50, alpha=0.7, label='Cross-Entropy', edgecolor='black')
plt.xlabel('Confidence (Max Probability)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Cross-Entropy Confidence Distribution', fontsize=13, fontweight='bold')
plt.axvline(conf_ce.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {conf_ce.mean():.3f}')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(conf_mse, bins=50, alpha=0.7, label='MSE', color='orange', edgecolor='black')
plt.xlabel('Confidence (Max Probability)', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('MSE Confidence Distribution', fontsize=13, fontweight='bold')
plt.axvline(conf_mse.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {conf_mse.mean():.3f}')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Mean confidence - Cross-Entropy: {conf_ce.mean():.4f}")
print(f"Mean confidence - MSE: {conf_mse.mean():.4f}")
print(f"\nCross-entropy tends to produce more confident predictions!")

## Part 4: Label Smoothing

### What is Label Smoothing?

Label smoothing is a regularization technique that prevents the model from becoming overconfident. Instead of using hard targets (0 or 1), we smooth them:

**Hard labels:** $[0, 0, 1, 0, 0]$ (for class 2 out of 5 classes)

**Smoothed labels (α=0.1):**
$$y_i^{smooth} = \begin{cases}
1 - \alpha & \text{if } i = \text{true class} \\
\frac{\alpha}{K-1} & \text{otherwise}
\end{cases}$$

Example with α=0.1: $[0.025, 0.025, 0.9, 0.025, 0.025]$

**Benefits:**
1. Prevents overconfident predictions
2. Improves model calibration
3. Can improve generalization
4. Acts as a regularizer

### Implement Label Smoothing

In [ ]:
class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1, num_classes=10):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes
        self.confidence = 1.0 - smoothing
    
    def forward(self, pred, target):
        """
        Args:
            pred: logits from model (batch_size, num_classes)
            target: ground truth labels (batch_size,)
        """
        # Convert to log probabilities
        pred = F.log_softmax(pred, dim=1)
        
        # Create smoothed labels
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (self.num_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        
        # Compute loss
        return torch.mean(torch.sum(-true_dist * pred, dim=1))

# Test the loss function
loss_fn_smooth = LabelSmoothingCrossEntropy(smoothing=0.1, num_classes=10)
test_logits = torch.randn(4, 10)
test_targets = torch.tensor([0, 1, 2, 3])
test_loss = loss_fn_smooth(test_logits, test_targets)
print(f"Test loss with label smoothing: {test_loss.item():.4f}")

### Visualize Label Smoothing Effect

In [ ]:
# Example: 5-class classification, true class is index 2
num_classes = 5
true_class = 2

# Hard labels
hard_labels = np.zeros(num_classes)
hard_labels[true_class] = 1.0

# Smoothed labels with different alpha values
alphas = [0.05, 0.1, 0.2]
smoothed_labels = []

for alpha in alphas:
    labels = np.full(num_classes, alpha / (num_classes - 1))
    labels[true_class] = 1.0 - alpha
    smoothed_labels.append(labels)

# Plot
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
x = np.arange(num_classes)
width = 0.6

# Hard labels
axes[0].bar(x, hard_labels, width, color='steelblue', edgecolor='black', linewidth=1.5)
axes[0].set_xlabel('Class', fontsize=11)
axes[0].set_ylabel('Probability', fontsize=11)
axes[0].set_title('Hard Labels (α=0)', fontsize=12, fontweight='bold')
axes[0].set_ylim([0, 1.1])
axes[0].grid(True, alpha=0.3, axis='y')

# Smoothed labels
for i, (alpha, labels) in enumerate(zip(alphas, smoothed_labels)):
    axes[i+1].bar(x, labels, width, color='coral', edgecolor='black', linewidth=1.5)
    axes[i+1].set_xlabel('Class', fontsize=11)
    axes[i+1].set_ylabel('Probability', fontsize=11)
    axes[i+1].set_title(f'Smoothed Labels (α={alpha})', fontsize=12, fontweight='bold')
    axes[i+1].set_ylim([0, 1.1])
    axes[i+1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Notice how label smoothing:")
print("1. Reduces the target probability for the true class")
print("2. Distributes some probability mass to other classes")
print("3. Higher α means more aggressive smoothing")

### Train with Different Smoothing Values

In [ ]:
# Train with different smoothing values
smoothing_values = [0.0, 0.1, 0.2]
results = {}

for smoothing in smoothing_values:
    print(f"\nTraining with label smoothing α={smoothing}...\n")
    
    model = CNNClassifier()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    if smoothing == 0.0:
        loss_fn = nn.CrossEntropyLoss()
    else:
        loss_fn = LabelSmoothingCrossEntropy(smoothing=smoothing, num_classes=10)
    
    train_losses, test_losses, train_accs, test_accs = train_mnist_model(
        model, train_loader, test_loader, loss_fn, optimizer, 
        epochs=10, use_softmax=False, device=device
    )
    
    results[smoothing] = {
        'model': model,
        'train_losses': train_losses,
        'test_losses': test_losses,
        'train_accs': train_accs,
        'test_accs': test_accs
    }

### Compare Label Smoothing Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Test loss
for smoothing, data in results.items():
    axes[0].plot(data['test_losses'], label=f'α={smoothing}', linewidth=2, marker='o')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Test Loss with Label Smoothing', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Test accuracy
for smoothing, data in results.items():
    axes[1].plot(data['test_accs'], label=f'α={smoothing}', linewidth=2, marker='o')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Test Accuracy with Label Smoothing', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nFinal Test Accuracy:")
for smoothing, data in results.items():
    print(f"α={smoothing}: {data['test_accs'][-1]:.4f}")

### Analyze Confidence with Label Smoothing

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (smoothing, data) in enumerate(results.items()):
    conf = get_confidence_distribution(data['model'], test_loader, device)
    
    axes[idx].hist(conf, bins=50, alpha=0.7, edgecolor='black')
    axes[idx].set_xlabel('Confidence (Max Probability)', fontsize=12)
    axes[idx].set_ylabel('Frequency', fontsize=12)
    axes[idx].set_title(f'Confidence Distribution (α={smoothing})', fontsize=13, fontweight='bold')
    axes[idx].axvline(conf.mean(), color='red', linestyle='--', linewidth=2, 
                      label=f'Mean: {conf.mean():.3f}')
    axes[idx].legend(fontsize=11)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("1. Label smoothing reduces average confidence")
print("2. This prevents overconfident (mis)predictions")
print("3. The distribution becomes more spread out with higher α")

### Calibration Analysis

Calibration measures how well predicted probabilities match actual correctness rates. A well-calibrated model that predicts 80% confidence should be correct 80% of the time.

In [ ]:
def compute_calibration(model, test_loader, device, n_bins=10):
    """Compute calibration curve data."""
    model.eval()
    confidences = []
    correctness = []
    
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            probs = F.softmax(outputs, dim=1)
            max_probs, preds = probs.max(dim=1)
            
            confidences.extend(max_probs.cpu().numpy())
            correctness.extend((preds == target).cpu().numpy())
    
    confidences = np.array(confidences)
    correctness = np.array(correctness)
    
    # Bin the predictions
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(confidences, bins) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)
    
    # Compute accuracy per bin
    bin_confidences = []
    bin_accuracies = []
    bin_counts = []
    
    for i in range(n_bins):
        mask = bin_indices == i
        if mask.sum() > 0:
            bin_confidences.append(confidences[mask].mean())
            bin_accuracies.append(correctness[mask].mean())
            bin_counts.append(mask.sum())
        else:
            bin_confidences.append(bins[i:i+2].mean())
            bin_accuracies.append(0)
            bin_counts.append(0)
    
    return bin_confidences, bin_accuracies, bin_counts

# Compute calibration for all models
plt.figure(figsize=(10, 8))

for smoothing, data in results.items():
    bin_conf, bin_acc, bin_counts = compute_calibration(data['model'], test_loader, device)
    plt.plot(bin_conf, bin_acc, 'o-', label=f'α={smoothing}', linewidth=2, markersize=8)

# Perfect calibration line
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect Calibration')

plt.xlabel('Confidence', fontsize=13)
plt.ylabel('Accuracy', fontsize=13)
plt.title('Calibration Curves (Reliability Diagram)', fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.tight_layout()
plt.show()

print("Interpretation:")
print("- Points closer to diagonal = better calibrated")
print("- Above diagonal = underconfident")
print("- Below diagonal = overconfident")
print("- Label smoothing typically improves calibration!")

## Part 5: Comprehensive Comparison

Let's create a summary table comparing all approaches.

In [ ]:
import pandas as pd

# Create comparison table
comparison_data = {
    'Loss Function': ['MSE', 'Cross-Entropy', 'CE + Label Smoothing (α=0.1)', 'CE + Label Smoothing (α=0.2)'],
    'Final Test Acc': [
        f"{test_accs_mnist_mse[-1]:.4f}",
        f"{results[0.0]['test_accs'][-1]:.4f}",
        f"{results[0.1]['test_accs'][-1]:.4f}",
        f"{results[0.2]['test_accs'][-1]:.4f}"
    ],
    'Mean Confidence': [
        f"{get_confidence_distribution(model_mnist_mse, test_loader, device).mean():.4f}",
        f"{get_confidence_distribution(results[0.0]['model'], test_loader, device).mean():.4f}",
        f"{get_confidence_distribution(results[0.1]['model'], test_loader, device).mean():.4f}",
        f"{get_confidence_distribution(results[0.2]['model'], test_loader, device).mean():.4f}"
    ],
    'Convergence': ['Slower', 'Fast', 'Fast', 'Fast'],
    'Calibration': ['Poor', 'Good', 'Better', 'Best'],
    'Use Case': [
        'Not recommended',
        'Standard classification',
        'When calibration matters',
        'Strong regularization needed'
    ]
}

df = pd.DataFrame(comparison_data)
print("\n" + "="*100)
print("COMPREHENSIVE COMPARISON")
print("="*100)
print(df.to_string(index=False))
print("="*100)

## Part 6: Key Takeaways and Best Practices

### Summary

1. **MSE for Classification:**
   - Can work but is suboptimal
   - Slower convergence due to gradient vanishing
   - Not probabilistically motivated
   - Generally not recommended

2. **Cross-Entropy Loss:**
   - Standard choice for classification
   - Fast convergence with strong gradients
   - Probabilistically principled
   - Can lead to overconfident predictions

3. **Label Smoothing:**
   - Prevents overconfidence
   - Improves calibration significantly
   - Acts as regularizer
   - May slightly reduce top-1 accuracy but improves robustness
   - Typical α values: 0.05 - 0.2

### Best Practices

- **Use Cross-Entropy** as your default loss for classification
- **Add label smoothing (α≈0.1)** when:
  - Calibration is important (e.g., medical diagnosis, autonomous driving)
  - You need uncertainty estimates
  - Model tends to overfit
  - You want more robust predictions
- **Avoid MSE** for classification unless you have specific reasons
- **Monitor calibration** not just accuracy

### Further Exploration

Try these experiments:
1. Test on imbalanced datasets
2. Compare with focal loss
3. Vary smoothing parameter α
4. Test on other datasets (CIFAR-10, ImageNet)
5. Combine with other regularization techniques

## Reflection Questions

1. Why does cross-entropy produce larger gradients for wrong predictions compared to MSE?
2. How does label smoothing act as a regularizer?
3. When might you want to use higher smoothing values (α > 0.2)?
4. Can you think of scenarios where overconfident predictions are dangerous?
5. How would you implement temperature scaling for calibration?

## Challenge: Implement Focal Loss

Focal loss is another alternative for handling classification, especially useful for imbalanced datasets:

$$L_{focal} = -\alpha(1-p_t)^{\gamma} \log(p_t)$$

Try implementing it and comparing with the approaches above!

In [ ]:
# Your focal loss implementation here
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        # TODO: Implement focal loss
        pass